In [246]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import date
import os

In [247]:
# 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [248]:
# 데이터 파일 경로
file_path = 'C:\\ai_x\\source\\JikFam\\data\\양파_이상치제거_주간기준_등급코드.csv'
output_dir = 'C:\\ai_x\\source\\JikFam\\eda_results'
os.makedirs(output_dir, exist_ok=True)

In [249]:
# 데이터 불러오기
try:
    df = pd.read_csv(file_path, encoding='cp949')
except FileNotFoundError:
    print(f"오류: 파일 '{file_path}'을(를) 찾을 수 없습니다.")
    # exit() 대신 노트북 환경에 맞게 중단 메시지 출력
    raise

In [250]:
print("--- 기초 통계량 ---")
print(df.describe())
print("--- 데이터 정보 ---")
df.info()

--- 기초 통계량 ---
           품목코드           품종코드           등급코드        총금액(원)       총거래량(kg)  \
count  931849.0  931849.000000  931849.000000  9.318490e+05  931849.000000   
mean        1.0      33.182620      12.243029  3.720260e+06    4379.534212   
std         0.0      44.904326       1.451906  7.913970e+06    8769.430982   
min         1.0       0.000000      11.000000  1.400000e+02       0.100000   
25%         1.0       1.000000      12.000000  3.216000e+05     540.000000   
50%         1.0       3.000000      12.000000  1.266500e+06    1800.000000   
75%         1.0      99.000000      12.000000  3.999500e+06    4905.000000   
max         1.0      99.000000      20.000000  6.000000e+08  875000.000000   

             평균단가(원)      주간평균단가(원)         직팜산지코드  휴일여부  명절지수  작기정보  \
count  931849.000000  931849.000000  931849.000000   0.0   0.0   0.0   
mean      836.443960     834.315277    1130.741654   NaN   NaN   NaN   
std       467.340121     339.923887     153.146112   NaN   NaN   N

In [251]:
# 데이터 결측치 확인
print("--- 결측치 확인 ---")
print(df.isnull().sum())

--- 결측치 확인 ---
주차                   0
연월일                  0
품목코드                 0
품목명                  0
품종코드                 0
품종명                  0
등급코드                 0
등급이름                 0
총금액(원)               0
총거래량(kg)             0
평균단가(원)              0
주간평균단가(원)            0
직팜산지코드               0
휴일여부            931849
명절지수            931849
작기정보            931849
일평균기온            27464
최고기온             27464
최저기온             27464
평균상대습도           27464
강수량(mm)          27464
1시간최고강수량(mm)     27464
dtype: int64


In [252]:
# 1. 주차 시작일 추출 ('YYYY-MM-DD~YYYY-MM-DD' → 'YYYY-MM-DD')
df['주차시작일'] = df['주차'].str.extract(r'^(\d{4}-\d{2}-\d{2})')
df['주차시작일'] = pd.to_datetime(df['주차시작일'], format='%Y-%m-%d', errors='coerce')

# 2. ISO 캘린더 연도 및 주차 번호 추출
iso = df['주차시작일'].dt.isocalendar()
df['주차_연도'] = iso.year
df['주차_번호'] = iso.week


In [253]:
df[['주차', '주차시작일', '주차_연도', '주차_번호']].head()


,주차,주차시작일,주차_연도,주차_번호
0,2018-01-01~2018-01-07,2018-01-01,2018,1
1,2018-01-01~2018-01-07,2018-01-01,2018,1
2,2018-01-01~2018-01-07,2018-01-01,2018,1
3,2018-01-01~2018-01-07,2018-01-01,2018,1
4,2018-01-01~2018-01-07,2018-01-01,2018,1


In [254]:
from datetime import date

def get_weeks_in_year(year):
    return date(year, 12, 28).isocalendar()[1]

summary = []

for crop in df['품목명'].unique():
    df_crop = df[df['품목명'] == crop].copy()

    # 🔒 2018~2024년까지만 필터
    df_crop = df_crop[(df_crop['주차_연도'] >= 2018) & (df_crop['주차_연도'] <= 2024)]

    # 실제 등장한 주차
    actual_weeks = set(zip(df_crop['주차_연도'], df_crop['주차_번호']))

    # 해당 연도 전체 주차 예상 목록
    expected_weeks = set()
    for y in range(2018, 2025):  # ✅ 2024까지!
        for w in range(1, get_weeks_in_year(y) + 1):
            expected_weeks.add((y, w))

    # 누락 주차 계산
    missing_weeks = sorted(list(expected_weeks - actual_weeks))

    # 결측치 / 이상치
    null_rows = df_crop.isnull().any(axis=1).sum()
    outlier_rows = df_crop[
        (df_crop['평균단가(원)'] > 6000) | (df_crop['평균단가(원)'] <= 0)
    ].shape[0]

    summary.append({
        '품목명': crop,
        '존재_연도범위': "2018~2024",  # 🔒 고정
        '예상_주차수': len(expected_weeks),
        '실제_주차수': len(actual_weeks),
        '누락_주차수': len(missing_weeks),
        '결측치_행수': null_rows,
        '단가_이상치_행수': outlier_rows,
        '누락_주차': missing_weeks
    })

df_summary = pd.DataFrame(summary)


In [255]:
df_summary[['품목명', '예상_주차수', '실제_주차수', '누락_주차수']]
# df_summary['누락_주차'][0]  # 누락된 주차 리스트


,품목명,예상_주차수,실제_주차수,누락_주차수
0,양파,365,365,0


In [256]:
len(missing_weeks)



0

In [257]:
def format_weeks(week_list):
    return [f"{y}년도 {w}주차" for (y, w) in week_list]

summary = []

for crop in df['품목명'].unique():
    df_crop = df[df['품목명'] == crop].copy()

    # 이상치 조건
    is_outlier = (df_crop['평균단가(원)'] > 6000) | (df_crop['평균단가(원)'] <= 0)
    df_outlier = df_crop[is_outlier]

    # 이상치가 발견된 (연도, 주차번호)만 추출
    outlier_weeks = sorted(set(zip(df_outlier['주차_연도'], df_outlier['주차_번호'])))
    outlier_weeks_str = format_weeks(outlier_weeks)

    summary.append({
        '품목명': crop,
        '이상치_주차수': len(outlier_weeks),
        '이상치_주차': outlier_weeks_str
    })

df_outlier_summary = pd.DataFrame(summary)


In [258]:
df_outlier_summary[['품목명', '이상치_주차수']]


,품목명,이상치_주차수
0,양파,33


In [259]:
# 품목별 누락 주차를 사람이 읽을 수 있는 형식으로 변환
def format_weeks(week_list):
    return [f"{y}년도 {w}주차" for (y, w) in week_list]

# 예: 배추 품목 기준으로 확인
# crop_name = '배추'

missing_weeks_raw = df_summary.loc[df_summary['품목명'] == crop_name, '누락_주차'].values[0]
missing_weeks_str = format_weeks(missing_weeks_raw)

# 결과 출력
for s in missing_weeks_str:
    print(s)


IndexError: index 0 is out of bounds for axis 0 with size 0